# SpecX — pipeline walkthrough

This notebook runs each stage of the pipeline on a service manual so you can inspect intermediate output:

1. **Parse** — tables first, prose outside tables
2. **Clean + structure** — learned boilerplate, headings
3. **Chunk** — table rows and heading-bounded prose
4. **Index + retrieve** — BGE + BM25 → RRF → cross-encoder
5. **Extract** — LLM → Pydantic → grounding check → JSON / CSV

Put the manual at `data/f150-workshop-manual.pdf` (or change `PDF` below) and set `OPENROUTER_API_KEY` in `.env`.

In [ ]:
import os, sys, json
ROOT = os.path.abspath('..')
os.chdir(ROOT)
sys.path.insert(0, os.path.join(ROOT, 'src'))
PDF = 'data/f150-workshop-manual.pdf'

## 1. Parse one page
Page 636 is the *Front Disc Brake* specifications page: three tables (materials, general specs, torque).

In [ ]:
import pymupdf
from specx.ingest.parse import parse_page

doc = pymupdf.open(PDF)
page = parse_page(doc[635])
for t in page.tables:
    print('TABLE', t.title)
    for row in t.rows[:5]:
        print('   ', row)
print('\nPROSE LINES (outside tables):')
for l in page.lines[:8]:
    print(f'   y={l.y0:5.0f} size={l.size} bold={l.bold} | {l.text}')

## 2–3. Clean, structure and chunk the whole manual
Boilerplate is learned from the document itself: repeated headers/footers, TOC/index pages, figure codes.

In [ ]:
from collections import Counter
from specx.ingest.chunk import build_chunks

chunks, stats = build_chunks(PDF, 'demo')
print(stats)
print(Counter(c.chunk_type for c in chunks), '| with values:', sum(c.has_spec for c in chunks))

for c in chunks:
    if c.page == 636 and c.chunk_type == 'table':
        print('\n---', c.heading_path)
        print(c.text)

In [ ]:
# A prose chunk: numbered steps stay together with their torque values
for c in chunks:
    if c.chunk_type == 'text' and 'Tighten the nuts to 55 Nm' in c.text:
        print(c.heading_path)
        print(c.text)
        break

## 4. Index and retrieve
`ingest()` runs the stages above, embeds the chunks and persists FAISS + chunks (cached by file hash).

In [ ]:
from specx import pipeline
from specx.query.retrieve import retrieve

info = pipeline.ingest(PDF)
index = pipeline.load_index(info.doc_id)

q = 'Torque for brake caliper bolts'
for h in retrieve(index, q):
    print(f'{h.score:6.2f}  D={h.dense_rank}  S={h.sparse_rank}  p.{h.chunk.page:<4} {h.chunk.chunk_type:<5} {h.chunk.text[:90]!r}')

## 5. Extract structured specs

In [ ]:
result = pipeline.query(info.doc_id, q)
print('mode:', result.mode, '| timings (ms):', result.timings_ms, '| rejected:', len(result.rejected))
print(json.dumps([s.required_fields() for s in result.specs], indent=2))

In [ ]:
# Full records with provenance
for s in result.specs:
    print(f'{s.component:<40} {s.value:>6} {s.unit or "":<4} ({s.alt_value} {s.alt_unit})  p.{s.page}  conf={s.confidence}  quote={s.source_quote!r}')

In [ ]:
# A few more spec types
for q in ['front brake fluid capacity', 'minimum brake disc thickness', 'stabilizer bar link nut torque']:
    r = pipeline.query(info.doc_id, q)
    print('\n#', q)
    for s in r.specs:
        print('   ', json.dumps(s.required_fields()), f'p.{s.page}')

In [ ]:
# CSV export
import csv, io
buf = io.StringIO()
w = csv.DictWriter(buf, fieldnames=['component', 'spec_type', 'value', 'unit'])
w.writeheader()
w.writerows(s.required_fields() for s in result.specs)
print(buf.getvalue())